In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('../data/Titanic Dataset.csv')

In [3]:
# wybieramy tylko sensowne zmienne kategoryczne
cat_cols = ['sex', 'pclass', 'embarked']

num_cols = df.select_dtypes(include=['number']).columns

print("PROCENTY BRAKÓW")

for num_col in num_cols:
    if df[num_col].isnull().sum() > 0:
        print(f"\n=== Analiza braków w kolumnie '{num_col}' ===")
        for cat_col in cat_cols:
            grouped = df.groupby(cat_col)[num_col].apply(lambda x: x.isnull().mean() * 100)
            print(grouped)


PROCENTY BRAKÓW

=== Analiza braków w kolumnie 'age' ===
sex
female    16.738197
male      21.945433
Name: age, dtype: float64
pclass
1    12.074303
2     5.776173
3    29.337094
Name: age, dtype: float64
embarked
C    21.481481
Q    59.349593
S    14.442013
Name: age, dtype: float64

=== Analiza braków w kolumnie 'fare' ===
sex
female    0.000000
male      0.118624
Name: fare, dtype: float64
pclass
1    0.000000
2    0.000000
3    0.141044
Name: fare, dtype: float64
embarked
C    0.000000
Q    0.000000
S    0.109409
Name: fare, dtype: float64

=== Analiza braków w kolumnie 'body' ===
sex
female    98.283262
male      86.595492
Name: body, dtype: float64
pclass
1    89.164087
2    88.808664
3    92.242595
Name: body, dtype: float64
embarked
C    90.740741
Q    94.308943
S    90.262582
Name: body, dtype: float64


In [4]:
df['sex'] = df['sex'].map({'male': 1, 'female': 0}) # od razu

In [5]:
missing_percentage = df.isnull().mean() * 100
print(missing_percentage.sort_values(ascending=False))

body         90.756303
cabin        77.463713
boat         62.872422
home.dest    43.086325
age          20.091673
embarked      0.152788
fare          0.076394
sibsp         0.000000
name          0.000000
survived      0.000000
pclass        0.000000
sex           0.000000
parch         0.000000
ticket        0.000000
dtype: float64


In [6]:
print(df['boat'].value_counts()) # nie ma sensu tutaj zamieniać braków modą

boat
13         39
C          38
15         37
14         33
4          31
10         29
5          27
3          26
9          25
11         25
8          23
16         23
7          23
D          20
6          20
12         19
2          13
A          11
B           9
1           5
13 15       2
5 7         2
C D         2
5 9         1
8 10        1
13 15 B     1
15 16       1
Name: count, dtype: int64


In [7]:
df_base = df.copy()
cols_to_drop_1 = ['name', 'body', 'cabin', 'boat', 'home.dest', 'ticket']

df_base['age'] = df_base['age'].fillna(df_base['age'].mean())
df_base['fare'] = df_base['fare'].fillna(df_base['fare'].mean())
df_base['embarked'] = df_base['embarked'].fillna(df_base['embarked'].mode()[0])

df_base.drop(columns=cols_to_drop_1, inplace=True)

# One-hot encoding
cat_features = df_base.select_dtypes(exclude=['number']).columns
df_base = pd.get_dummies(df_base, columns=cat_features, drop_first=True, dtype=int)

print(f"Pozostałe braki w df_base: {df_base.isnull().sum().sum()}")

Pozostałe braki w df_base: 0


In [8]:
# display(df_base)

In [9]:
df_adv = df.copy()

df_adv['has_cabin'] = df_adv['cabin'].notnull().astype(int)

# Wyodrębnienie tytułu z imienia
df_adv['title'] = df_adv['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

rare_titles = df_adv['title'].value_counts()[df_adv['title'].value_counts() < 10].index
df_adv['title'] = df_adv['title'].replace(list(rare_titles), 'Rare')

# Uzupełnianie wieku medianą w zależności od tytułu
median_by_title = df_adv.groupby('title')['age'].median()
df_adv['age'] = df_adv.apply(
    lambda row: median_by_title[row['title']] if pd.isnull(row['age']) else row['age'],
    axis=1
)
# Jeśli coś jeszcze zostało puste, uzupełnij medianą ogólną
df_adv['age'] = df_adv['age'].fillna(df_adv['age'].median())

# Pozostałe braki
df_adv['fare'] = df_adv['fare'].fillna(df_adv['fare'].mean())
df_adv['embarked'] = df_adv['embarked'].fillna(df_adv['embarked'].mode()[0])

# Usuwanie zbędnych kolumn
cols_to_drop_2 = ['name', 'body', 'boat', 'home.dest', 'ticket', 'cabin']
df_adv.drop(columns=cols_to_drop_2, inplace=True)

# One-hot encoding
cat_features = df_adv.select_dtypes(exclude=['number']).columns
df_adv = pd.get_dummies(df_adv, columns=cat_features, drop_first=True, dtype=int)

print(f"Pozostałe braki w df_adv: {df_adv.isnull().sum().sum()}")

Pozostałe braki w df_adv: 0


In [10]:
# display(df_adv)

In [11]:
def evaluate_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return mae, r2

In [12]:
X_base = df_base.drop('survived', axis=1)
y_base = df_base['survived']

X_adv = df_adv.drop('survived', axis=1)
y_adv = df_adv['survived']

mae_base, r2_base = evaluate_model(X_base, y_base)
mae_adv, r2_adv = evaluate_model(X_adv, y_adv)

In [13]:
print("\n=== WYNIKI REGRESJI LINOWEJ ===")
print(f"Baseline model: MAE = {mae_base:.4f}, R2 = {r2_base:.4f}")
print(f"Advanced model: MAE = {mae_adv:.4f}, R2 = {r2_adv:.4f}")


=== WYNIKI REGRESJI LINOWEJ ===
Baseline model: MAE = 0.3085, R2 = 0.3000
Advanced model: MAE = 0.3031, R2 = 0.3177


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def evaluate_model_log(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

In [15]:
acc_base = evaluate_model_log(X_base, y_base)
acc_adv = evaluate_model_log(X_adv, y_adv)

print("\n=== WYNIKI REGRESJI LOGISTYCZNEJ ===")
print(f"Baseline model: Accuracy = {acc_base:.4f}")
print(f"Advanced model: Accuracy = {acc_adv:.4f}")


=== WYNIKI REGRESJI LOGISTYCZNEJ ===
Baseline model: Accuracy = 0.7672
Advanced model: Accuracy = 0.7977


Na internecje znalazłem, że wynik >80 są juś uznawane za bardzo dobre dla tego dataseta, więc skoro taka porsta regresja jest tak blisko to chyba nieźle